# 🎵 YuE2｜Colab 歌曲生成 + 自動 LRC（v9）

在 Colab 上執行 m-a-p 的開源歌曲模型 **YuE2**，附批量生成、Seed 管理、LRC 歌詞時間軸、MP3 輸出，以及 Google Drive 模型備份。

## 使用方式

1. 「檔案 → 在雲端硬碟中儲存複本」，之後都從自己的副本開啟
2. 「執行階段 → 變更執行階段類型」選 **A100**（官方建議 24GB 以上、支援 BF16 的 GPU）
3. 修改 **③ 歌曲設定**，在 **③-2** 貼上 STYLE 和 LYRICS
4. 有需要時才調整 **③-3 進階採樣**，或在 **③-4** 輸入／載入 ABC 樂譜
5. 「執行階段 → 全部執行」
6. 想用**圖形介面**：保持 ⑤G 的 `USE_GUI` 勾選，「全部執行」後到 ⑤G 的輸出找網址和帳號密碼；此時 ⑤ 表單批量會自動略過。想用原本的表單流程就取消勾選。

- ① 若自動重新啟動執行階段，再按一次「全部執行」即可（第二次會直接略過安裝）。
- 同一個 Runtime 換歌：改完 ③／③-2／③-3／③-4 後直接執行 ⑤（需先在本 Runtime 執行過各設定格）；④ 偵測到模型已載入會自動略過。
- ⑤B、⑦ 預設不執行，需要時勾選後再單獨執行。

## 寫法重點

| 項目 | 說明 |
|---|---|
| STYLE | 語言、曲風、樂器、人聲、速度等，用逗號或換行分隔，可以寫短句 |
| LYRICS | `[Verse]` `[Chorus]` 等段落標籤可以保留，產生 LRC 時會自動略過 |
| LRC | 預設先用 Demucs 分離人聲再對歌詞（直接用完整混音常常一行都對不上）。第一次用到會自動安裝，約多花 1 分鐘；不需要可取消 `LRC_SEPARATE_VOCALS`。狀態會顯示實際對上幾行。 |
| 長度 | `MAX_DURATION_SEC` 為最長秒數（1～900），不是保證唱滿；`0` 使用官方預設上限（約 360 秒）。若歌詞或 ABC 規劃占用上下文，會安全下修 token 預算並顯示提示。 |
| 進階採樣 | ③-3 將 ABC 與 Music 的 Temperature、Top P、Top K、Repetition Penalty、Penalty Window、Min Tokens 獨立調整；ABC Max Tokens 也是獨立設定。預設值與官方 Python Pipeline 一致。 |
| 手動 ABC | ③-4 選 `paste` 直接貼樂譜，或選 `file` 指定既有 `score.abc` 路徑；`auto` 保持自動規劃。手動樂譜不能搭配 COT=`off`。 |
| Music Max Tokens | 一般使用 `MAX_DURATION_SEC` 控制；進階可設定 `MUSIC_MAX_TOKENS_OVERRIDE`。**兩者不要同時自訂**。實際 token 預算仍受上下文限制。 |
| ODE Steps | 預設 32，數值增加不保證更好，只會增加音訊合成運算。 |

## 模型備份

勾選 ① 的 `USE_DRIVE_BACKUP` 後，最後一格 ⑧ 會在第一次成功生成後，把程式碼和模型（含 VAE）備份到 Google Drive。
平常仍從 Hugging Face／GitHub 下載（通常比 Drive 快）；只有它們無法使用時，才會自動改用 Drive 備份。

> YuE2-3B 權重採 CC-BY-NC-4.0 授權，不可商業使用。

## 授權與致謝

- 本 notebook 的程式碼以 MIT License 釋出。
- 執行時會從官方來源安裝 [YuE2](https://github.com/multimodal-art-projection/YuE)（程式碼 Apache-2.0）與 [lyric-align](https://github.com/ijuinryukichi/lyric-align)（MIT），本 notebook 不包含它們的程式碼。
- 模型權重（YuE2、Whisper 等）各有自己的授權條款，使用前請自行確認。

In [ ]:
# @title ① 🔧 環境檢查與安裝
# 核對 YuE2 官方要求的套件版本；同一個 Runtime 版本正確時直接略過。
# 若安裝前已載入舊版套件，會自動重新啟動執行階段，之後再按一次「全部執行」即可。

import os
import sys
import shutil
import signal
import subprocess
import importlib
import importlib.metadata
from pathlib import Path

try:
    from packaging.version import Version
except ImportError:
    from pip._vendor.packaging.version import Version

# @markdown ### 💾 Google Drive 備份
USE_DRIVE_BACKUP = True  # @param {type:"boolean"}
BACKUP_ROOT = "/content/drive/MyDrive/AI模型備份/YuE2"  # @param {type:"string"}

# @markdown ### 進階（平常不用勾）
FORCE_UPDATE_YUE2 = False  # @param {type:"boolean"}
# @markdown 需要重啟時自動重新啟動執行階段
AUTO_RESTART = True  # @param {type:"boolean"}

YUE_REPO = Path("/content/YuE")
YUE_REPO_URL = "https://github.com/multimodal-art-projection/YuE.git"
CONSTRAINTS = Path("/content/yue2_constraints.txt")
YUE2_RESTORED = set()

# yue2-infer 0.1.6 的固定依賴
REQUIRED = {
    "yue2-infer": "0.1.6",
    "torch": "2.10.0",
    "transformers": "4.57.6",
    "huggingface-hub": "0.36.2",
    "safetensors": "0.7.0",
    "accelerate": "1.13.0",
    "numpy": "2.2.6",
    "tiktoken": "0.12.0",
    "soundfile": "0.13.1",
}
SMALL_CORE = [p for p in REQUIRED if p not in ("yue2-infer", "torch")]
TORCH_COMPANIONS = ["torchvision", "torchaudio"]
HEAVY_MODULES = ["torch", "transformers", "yue2", "numpy", "safetensors", "accelerate",
                 "huggingface_hub", "tiktoken", "soundfile", "torchvision", "torchaudio"]
preloaded = [m for m in HEAVY_MODULES if m in sys.modules]


def run_cmd(cmd, check=True):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(c) for c in cmd], check=check)


def pip_install(*args):
    # 所有安裝都套用 constraints，避免核心版本被其他套件升級
    return run_cmd([sys.executable, "-m", "pip", "install", "-c", str(CONSTRAINTS), *args])


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def version_ok(pkg):
    ver = package_version(pkg)
    return ver is not None and Version(ver).public == Version(REQUIRED[pkg]).public


def fresh_python(code):
    r = subprocess.run([sys.executable, "-c", code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0, r.stdout


def fresh_yue_import_test():
    # 用全新的 process 測試，避免目前 kernel 已載入舊版造成誤判
    return fresh_python(
        "import torch; print('torch', torch.__version__)\n"
        "import transformers; print('transformers', transformers.__version__)\n"
        "from transformers import GenerationMixin, PretrainedConfig, PreTrainedModel\n"
        "from yue2 import YuE2Pipeline; print('YuE2Pipeline OK')\n"
    )


def print_versions(title):
    print(f"===== {title} =====")
    for pkg, required in REQUIRED.items():
        print(f"{'✅' if version_ok(pkg) else '❌'} {pkg:18s}: {package_version(pkg)}（需要 {required}）")


# ---------- Google Drive 備份工具 ----------
import tarfile
from datetime import datetime

DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
BACKUP_DIR = Path(BACKUP_ROOT)

if USE_DRIVE_BACKUP and not DRIVE_MYDRIVE.exists():
    from google.colab import drive
    drive.mount("/content/drive")


def drive_ready() -> bool:
    return USE_DRIVE_BACKUP and DRIVE_MYDRIVE.exists()


def backup_done(name: str) -> bool:
    return drive_ready() and (BACKUP_DIR / name / ".complete").exists()


def dir_size_gb(p: Path) -> float:
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1024**3


def _begin(name: str) -> Path:
    tmp = BACKUP_DIR / f"{name}.partial"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    return tmp


def _finish(tmp: Path, name: str):
    final = BACKUP_DIR / name
    shutil.rmtree(final, ignore_errors=True)
    tmp.rename(final)
    (final / ".complete").write_text(datetime.now().isoformat(timespec="seconds"))


def backup_dirs(name: str, sources: dict):
    """把多個資料夾備份到 BACKUP_DIR/name/<子資料夾>。中途失敗不會留下「看起來完整」的備份。"""
    tmp = _begin(name)
    for sub, src in sources.items():
        print(f"   複製 {sub}（{dir_size_gb(src):.1f} GB）……")
        shutil.copytree(src, tmp / sub, symlinks=False)
    _finish(tmp, name)


def backup_code_tar(name: str, src: Path, exclude=()):
    """程式碼打包成 code.tar.gz（排除大型資料夾）。"""
    tmp = _begin(name)
    exclude = set(exclude) | {".git"}

    def keep(ti):
        parts = Path(ti.name).parts
        return None if len(parts) > 1 and parts[1] in exclude else ti

    with tarfile.open(tmp / "code.tar.gz", "w:gz") as tf:
        tf.add(src, arcname="code", filter=keep)
    _finish(tmp, name)


def restore_dirs(name: str, dst_parent: Path, only=None) -> bool:
    """從 Drive 把備份複製回本機（不要直接從 Drive 載入模型，會很慢）。"""
    if not backup_done(name):
        return False
    for sub in sorted((BACKUP_DIR / name).iterdir()):
        if not sub.is_dir() or (only and sub.name not in only):
            continue
        dst = Path(dst_parent) / sub.name
        print(f"   從 Drive 還原 {sub.name}（{dir_size_gb(sub):.1f} GB）……")
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(sub, dst)
    return True


def restore_code_tar(name: str, dst: Path) -> bool:
    if not backup_done(name):
        return False
    tmp = Path("/content/_code_restore")
    shutil.rmtree(tmp, ignore_errors=True)
    with tarfile.open(BACKUP_DIR / name / "code.tar.gz") as tf:
        try:
            tf.extractall(tmp, filter="data")
        except TypeError:
            tf.extractall(tmp)
    dst.mkdir(parents=True, exist_ok=True)
    for p in (tmp / "code").iterdir():
        target = dst / p.name
        if target.is_dir():
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()
        shutil.move(str(p), str(target))
    shutil.rmtree(tmp, ignore_errors=True)
    return True


if drive_ready():
    have = [p.name for p in BACKUP_DIR.glob("*") if (p / ".complete").exists()] if BACKUP_DIR.exists() else []
    print("💾 Drive 備份位置：", BACKUP_DIR, "｜已有：", have or "（尚無）")


CONSTRAINTS.write_text("".join(f"{p}=={v}\n" for p, v in REQUIRED.items() if p != "yue2-infer"))
print_versions("目前版本")
core_changed = False

# ---------- 1. YuE2 本體 ----------
if not version_ok("yue2-infer") or FORCE_UPDATE_YUE2:
    if not (YUE_REPO / "pyproject.toml").exists():
        shutil.rmtree(YUE_REPO, ignore_errors=True)
        r = run_cmd(["git", "clone", "--depth", "1", YUE_REPO_URL, YUE_REPO], check=False)
        if r.returncode != 0 or not (YUE_REPO / "pyproject.toml").exists():
            print("⚠️ GitHub 無法使用，改用 Drive 備份")
            shutil.rmtree(YUE_REPO, ignore_errors=True)
            if not restore_code_tar("yue2_code", YUE_REPO):
                raise RuntimeError("無法取得 YuE2 程式碼（GitHub 與 Drive 備份都不可用）。")
            YUE2_RESTORED.add("yue2_code")
    elif FORCE_UPDATE_YUE2 and (YUE_REPO / ".git").exists():
        run_cmd(["git", "-C", YUE_REPO, "pull", "--ff-only"], check=False)

    run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    pip_install(str(YUE_REPO))
    core_changed = True
else:
    print("✅ yue2-infer 版本正確")

# ---------- 2. torch ----------
if not version_ok("torch"):
    print(f"\n⚠️ torch {package_version('torch')} → {REQUIRED['torch']}")
    pip_install(f"torch=={REQUIRED['torch']}")
    core_changed = True

# ---------- 3. 其他核心套件（--no-deps，避免連帶升級 torch） ----------
small_mismatched = [p for p in SMALL_CORE if not version_ok(p)]
if small_mismatched:
    print("\n⚠️ 版本不符：", small_mismatched)
    pip_install("--force-reinstall", "--no-deps", "--no-cache-dir",
                *[f"{p}=={REQUIRED[p]}" for p in small_mismatched])
    core_changed = True

# ---------- 4. torchvision / torchaudio（YuE2 不需要；與 torch 不相容就移除） ----------
for companion in TORCH_COMPANIONS:
    if package_version(companion) is None:
        continue
    ok, log = fresh_python(f"import {companion}")
    if not ok:
        print(f"⚠️ {companion} 與 torch 不相容，移除：{log.strip().splitlines()[-1]}")
        run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", companion])
        core_changed = True

# ---------- 5. LRC 工具 ----------
if package_version("lyric-align") is None or package_version("faster-whisper") is None:
    pip_install("lyric-align[asr]")

av_ver = package_version("av")
if av_ver is None:
    pip_install("av<19")
elif Version(av_ver).major >= 19:
    pip_install("--no-deps", "av<19")      # PyAV 19 與 faster-whisper 不相容

# ---------- 6. 驗證 ----------
fresh_ok, fresh_log = fresh_yue_import_test()
if not fresh_ok:
    print("\n⚠️ import 測試失敗，重新安裝核心套件：\n" + fresh_log)
    pip_install("--force-reinstall", "--no-deps", "--no-cache-dir",
                *[f"{p}=={REQUIRED[p]}" for p in SMALL_CORE])
    core_changed = True
    fresh_ok, fresh_log = fresh_yue_import_test()

print("\n===== import 測試 =====\n" + fresh_log)
print_versions("最終版本")

still_wrong = [p for p in REQUIRED if not version_ok(p)]
if still_wrong:
    raise RuntimeError(f"安裝後版本仍不符：{still_wrong}。重跑結果會相同，請保留這格完整輸出以便排查。")
if not fresh_ok:
    raise RuntimeError("YuE2 import 失敗，請保留上方完整輸出以便排查。")

if core_changed and preloaded:
    msg = f"\n✅ 套件已更新，但目前 kernel 已載入舊版（{preloaded}），需要重新啟動執行階段。\n重新啟動後再按一次「全部執行」即可，已安裝的套件不會遺失。"
    if AUTO_RESTART:
        print(msg + "\n🔄 正在自動重新啟動……")
        try:
            from IPython import get_ipython
            get_ipython().kernel.do_shutdown(restart=True)
        except Exception:
            os.kill(os.getpid(), signal.SIGKILL)
        raise SystemExit
    raise RuntimeError(msg + "\n請點「執行階段 → 重新啟動執行階段」。")

importlib.invalidate_caches()
print("\n✅ 環境就緒，請繼續執行下一格。")


In [ ]:
# @title ② 📁 掛載 Google Drive + GPU 檢查
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

import torch

if not torch.cuda.is_available():
    raise RuntimeError("沒有偵測到 GPU，請到「執行階段 → 變更執行階段類型」選擇 GPU。")

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU：{torch.cuda.get_device_name(0)}（{gpu_mem_gb:.1f} GB）")
if gpu_mem_gb < 24:
    print("⚠️ YuE2 官方建議至少 24GB VRAM，目前顯存可能不足。")


In [ ]:
# @title ③ 🎛️ 歌曲設定
import re

# @markdown ### 💾 儲存位置
OUTPUT_ROOT = "/content/drive/MyDrive/AI音樂/YuE2"  # @param {type:"string"}
PROJECT_NAME = "我的歌曲"  # @param {type:"string"}
# @markdown 檔名前綴：`song_v` → song_v01、song_v02……
FILE_PREFIX = "song_v"  # @param {type:"string"}
VERSION_DIGITS = 2  # @param {type:"integer"}

# @markdown ---
# @markdown ### 🎲 批量與 Seed
BATCH_COUNT = 3  # @param {type:"slider", min:1, max:20, step:1}
# @markdown `random` 隨機｜`fixed` 固定｜`sequence` 從固定值逐首 +1｜`list` 依清單
SEED_MODE = "random"  # @param ["random", "fixed", "sequence", "list"]
FIXED_SEED = 123456  # @param {type:"integer"}
SEED_LIST_TEXT = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎼 YuE2
MODEL_ID = "m-a-p/YuE2-3B"  # @param {type:"string"}
# @markdown `full` 規劃旋律＋和弦（建議）｜`melody` 只規劃旋律｜`off` 不規劃
COT_MODE = "full"  # @param ["full", "melody", "off"]
# @markdown CFG：留空 = 官方預設；自訂可填 0～20（以官方 Python API 為準）
CFG_SCALE_TEXT = ""  # @param {type:"string"}
# @markdown 最長秒數：0 = 官方預設（約 360 秒／9000 tokens）；1～900 = 自訂上限，非保證長度
MAX_DURATION_SEC = 0  # @param {type:"integer"}

# @markdown ---
# @markdown ### 📝 LRC
MAKE_LRC = True  # @param {type:"boolean"}
LRC_LANGUAGE = "en"  # @param ["zh", "en", "ja"]
# @markdown 慢歌、長音多的歌建議勾選；Rap 等密集唱法可取消
LRC_NO_VAD = True  # @param {type:"boolean"}
# @markdown 對不上的行用推測時間補齊
LRC_INTERPOLATE = False  # @param {type:"boolean"}
LRC_WHISPER_MODEL = "medium"  # @param ["small", "medium", "large-v3"] {allow-input: true}
# @markdown `cpu` 最穩定；`cuda` 較快，失敗會自動改用 CPU
LRC_DEVICE = "cpu"  # @param ["cpu", "cuda"]
# @markdown 先用 Demucs 分離人聲再對歌詞，準確很多（第一次會自動安裝，約多花 1 分鐘）；取消則直接用完整混音
LRC_SEPARATE_VOCALS = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🎧 輸出（FLAC 母檔一定保留）
MAKE_MP3 = True  # @param {type:"boolean"}
MP3_BITRATE = "320k"  # @param ["192k", "256k", "320k"]


SEED_LIST = [int(s) for s in re.split(r"[,\s，、]+", SEED_LIST_TEXT.strip()) if s]
CFG_SCALE = float(CFG_SCALE_TEXT) if CFG_SCALE_TEXT.strip() else None
if CFG_SCALE is not None and not (0.0 <= CFG_SCALE <= 20.0):
    raise ValueError("CFG_SCALE 需介於 0～20；留空則使用官方預設。")
if not 0 <= MAX_DURATION_SEC <= 900:
    raise ValueError("MAX_DURATION_SEC 需為 0～900 秒（0 = 官方預設）。")
MAX_DURATION_SEC = MAX_DURATION_SEC if MAX_DURATION_SEC > 0 else None

if SEED_MODE == "list" and len(SEED_LIST) < BATCH_COUNT:
    raise ValueError(f"SEED_LIST 只有 {len(SEED_LIST)} 個，不夠生成 {BATCH_COUNT} 首。")

print("✅ 設定已讀取")


In [ ]:
# @title ③-2 ✍️ 風格與歌詞

# STYLE：語言、曲風、樂器、人聲、速度、氛圍等，逗號或換行分隔
STYLE = """
English,
warm acoustic pop,
approximately 90 BPM,
approximately 90 to 120 seconds,
single clear female lead vocal,
acoustic guitar,
soft piano,
light percussion,
gentle and hopeful,
memorable chorus,
begin with a short instrumental intro,
clean definite ending
""".strip()

# LYRICS：段落標籤可以保留
LYRICS = """
[Intro]

[Verse]
Morning light on an empty street
Coffee cups and a quiet beat
I hum a tune that I can't forget
A little song that isn't finished yet

[Chorus]
Sing it slow, sing it low
Let the melody take it home
Every word I couldn't say
Finds a way to you today

[Verse]
Paper notes on the kitchen wall
Half a rhyme and a melody call
I write it down so it won't fade
A tiny promise that the morning made

[Chorus]
Sing it slow, sing it low
Let the melody take it home
Every word I couldn't say
Finds a way to you today

[Outro]
""".strip()

print(f"專案：{PROJECT_NAME}｜批量：{BATCH_COUNT}｜Seed：{SEED_MODE}｜LRC：{LRC_LANGUAGE}")
print(f"STYLE {len(STYLE)} 字｜LYRICS {len(LYRICS.splitlines())} 行")


In [ ]:
# @title ③-3 🎚️ 進階採樣設定（不確定就保持預設）
# @markdown ABC：先規劃旋律與和弦（COT=off 或使用手動 ABC 時會略過）。
# @markdown ### 🎼 ABC 規劃階段
ABC_TEMPERATURE = 0.7  # @param {type:"number"}
ABC_TOP_P = 0.9  # @param {type:"number"}
ABC_TOP_K = 30  # @param {type:"integer"}
ABC_REPETITION_PENALTY = 1.005  # @param {type:"number"}
# @markdown 官方 Sampling 的 Penalty Window 合法範圍為 1～100
ABC_PENALTY_WINDOW = 100  # @param {type:"integer"}
ABC_MIN_TOKENS = 32  # @param {type:"integer"}
ABC_MAX_TOKENS = 4096  # @param {type:"integer"}

# @markdown ---
# @markdown ### 🎤 Music 歌曲生成階段
MUSIC_TEMPERATURE = 1.0  # @param {type:"number"}
MUSIC_TOP_P = 0.95  # @param {type:"number"}
MUSIC_TOP_K = 100  # @param {type:"integer"}
MUSIC_REPETITION_PENALTY = 1.2  # @param {type:"number"}
MUSIC_PENALTY_WINDOW = 50  # @param {type:"integer"}
MUSIC_MIN_TOKENS = 200  # @param {type:"integer"}
# @markdown 進階 Music 最大 token 數：`0` = 依 ③ 的 MAX_DURATION_SEC；非 0 時不能同時自訂秒數。
MUSIC_MAX_TOKENS_OVERRIDE = 0  # @param {type:"integer"}

# @markdown ---
# @markdown ### 🎧 音訊合成
# @markdown 官方預設 32；增加步數不一定改善品質，可能更慢。
ODE_STEPS = 32  # @param {type:"integer"}

print("✅ 進階設定已讀取（預設與官方值一致）")


In [ ]:
# @title ③-4 🎼 手動 ABC 樂譜（可略過）
# @markdown `auto` 自動規劃（預設）｜`paste` 使用下方 ABC 文字｜`file` 讀取已有的 .abc 檔案
ABC_SOURCE = "auto"  # @param ["auto", "paste", "file"]
# @markdown `file` 範例：`/content/drive/MyDrive/AI音樂/YuE2/我的歌曲/song_v01/score.abc`
ABC_FILE_PATH = ""  # @param {type:"string"}

# 要用 paste 時，把生成歌曲資料夾內的 score.abc 全文貼在三引號內，並把 ABC_SOURCE 改成 paste。
# 使用 file 時填入 ABC_FILE_PATH；此欄會被略過。
MANUAL_ABC_TEXT = r"""
""".strip()

print(f"🎼 ABC 來源：{ABC_SOURCE}" +
      (f"｜檔案：{ABC_FILE_PATH}" if ABC_SOURCE == "file" else ""))


In [ ]:
# @title 🧰 共用工具函式
import hashlib
import json
import random
import re
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

import soundfile as sf
import torch
from yue2 import YuE2Pipeline
from yue2.protocol import CONTEXT, make_request, token_prefixes

PROJECT_DIR = Path(OUTPUT_ROOT) / PROJECT_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


def next_version_number(root: Path, prefix: str) -> int:
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)$")
    found = [int(m.group(1)) for p in root.iterdir() if p.is_dir() and (m := pattern.match(p.name))]
    return max(found, default=0) + 1


def pick_seed(i: int) -> int:
    mode = SEED_MODE.lower().strip()
    if mode == "random":
        return random.SystemRandom().randint(1, 2_147_483_647)
    if mode == "fixed":
        return int(FIXED_SEED)
    if mode == "sequence":
        return int(FIXED_SEED) + i
    if mode == "list":
        return int(SEED_LIST[i])
    raise ValueError("SEED_MODE 必須是 random / fixed / sequence / list")


# YuE2 0.1.6 的 __call__(**kwargs) 支援 abc_sampling / semantic_sampling / abc / cfg_scale。
# 最長秒數約為 25 semantic tokens/秒；ABC 與 music 共享 24576 token 上下文。
# 裝載自訂 ABC 時採實際 token 數估算；自動 ABC 則保留規劃階段的 token 上限。
from dataclasses import replace
from yue2.protocol import Sampling


def resolve_manual_abc() -> str | None:
    source = ABC_SOURCE.strip().lower()
    if source == "auto":
        return None
    if source not in {"paste", "file"}:
        raise ValueError("ABC_SOURCE 僅接受 auto / paste / file")
    if COT_MODE == "off":
        raise ValueError("COT=off 不支援手動 ABC，請改 full/melody 或將 ABC_SOURCE 設為 auto。")
    if source == "paste":
        score = MANUAL_ABC_TEXT.strip()
    else:
        raw_path = ABC_FILE_PATH.strip()
        if not raw_path:
            raise ValueError("ABC_SOURCE=file 時須填寫 ABC_FILE_PATH。")
        file_path = Path(raw_path).expanduser()
        if not file_path.is_file():
            raise FileNotFoundError(f"找不到 ABC 檔案：{file_path}")
        score = file_path.read_text(encoding="utf-8-sig").strip()
    if not score:
        raise ValueError("手動 ABC 內容為空；請貼上樂譜或選擇其他檔案。")
    return score


def validate_sampling_options() -> None:
    # 使用官方 Sampling dataclass 驗證數值；例如 top_p > 0、penalty_window <= 100。
    Sampling(temperature=float(ABC_TEMPERATURE), top_p=float(ABC_TOP_P),
             top_k=int(ABC_TOP_K), repetition_penalty=float(ABC_REPETITION_PENALTY),
             penalty_window=int(ABC_PENALTY_WINDOW),
             min_tokens=int(ABC_MIN_TOKENS), max_tokens=int(ABC_MAX_TOKENS))
    # 此處以官方預設或使用者指定最大值作驗證，真正最大值會在 build_request 對上下文下修。
    raw_max = int(MUSIC_MAX_TOKENS_OVERRIDE)
    if not 0 <= raw_max <= 22500:
        raise ValueError("MUSIC_MAX_TOKENS_OVERRIDE 需為 0～22500（0 = 依 Max Duration）。")
    if raw_max and MAX_DURATION_SEC is not None:
        raise ValueError("MAX_DURATION_SEC 與 MUSIC_MAX_TOKENS_OVERRIDE 只能擇一設定；請將另一個設 0。")
    requested_max = raw_max or (round(MAX_DURATION_SEC * 25) if MAX_DURATION_SEC is not None else 9000)
    Sampling(temperature=float(MUSIC_TEMPERATURE), top_p=float(MUSIC_TOP_P),
             top_k=int(MUSIC_TOP_K), repetition_penalty=float(MUSIC_REPETITION_PENALTY),
             penalty_window=int(MUSIC_PENALTY_WINDOW),
             min_tokens=min(int(MUSIC_MIN_TOKENS), max(1, requested_max)),
             max_tokens=max(1, requested_max))
    if int(MUSIC_MIN_TOKENS) < 0:
        raise ValueError("MUSIC_MIN_TOKENS 需大於或等於 0。")
    if not 1 <= int(ODE_STEPS) <= 128:
        raise ValueError("ODE_STEPS 需為 1～128（官方預設 32）。")


def apply_pipeline_settings():
    # 使用者可只重跑 ③-3 + ⑤，不必重新從 Hugging Face 載入模型。
    steps = int(ODE_STEPS)
    if pipe.generation_config.ode_steps != steps:
        pipe.generation_config = replace(pipe.generation_config, ode_steps=steps)
        print(f"🎧 ODE Steps 已調整為 {steps}（官方預設 32）")


def build_request(song_id: str, seed: int) -> dict:
    request = {"style": STYLE, "lyrics": LYRICS, "cot": COT_MODE,
               "seed": seed, "id": song_id}
    if CFG_SCALE is not None:
        request["cfg_scale"] = float(CFG_SCALE)
    if ABC_OVERRIDE_TEXT is not None:
        request["abc"] = ABC_OVERRIDE_TEXT

    default_abc = pipe.generation_config.abc
    default_sem = pipe.generation_config.semantic
    abc_max = int(ABC_MAX_TOKENS)

    song_request = make_request(**request)
    if COT_MODE == "off":
        prefix_len = len(token_prefixes(song_request, pipe.tokenizer))
        abc_reserve = 0
    elif ABC_OVERRIDE_TEXT is not None:
        # 手動樂譜不生成 ABC，直接用實際 token 數計算最終音樂 prefix。
        abc_ids = pipe.tokenizer.encode(ABC_OVERRIDE_TEXT)
        prefix_len = len(token_prefixes(song_request, pipe.tokenizer, abc_ids))
        abc_reserve = 0
    else:
        # 自動 ABC 必須先完成規劃階段；其本身也受 context 限制。
        abc_plan_len = len(token_prefixes(song_request, pipe.tokenizer))
        abc_budget = CONTEXT - abc_plan_len - 1
        if abc_budget < 1:
            raise ValueError("STYLE／LYRICS 太長，無足夠的上下文生成 ABC。")
        abc_max = min(abc_max, abc_budget)
        if abc_max != int(ABC_MAX_TOKENS):
            print(f"⚠️ ABC token 上限：{ABC_MAX_TOKENS} → {abc_max}（上下文保護）")
        if abc_max < int(ABC_MIN_TOKENS):
            print(f"⚠️ ABC min_tokens 將縮到 {abc_max}（符合剩餘上下文）")
        prefix_len = len(token_prefixes(song_request, pipe.tokenizer, []))
        abc_reserve = abc_max
        request["abc_sampling"] = {
            "temperature": float(ABC_TEMPERATURE),
            "top_p": float(ABC_TOP_P),
            "top_k": int(ABC_TOP_K),
            "repetition_penalty": float(ABC_REPETITION_PENALTY),
            "penalty_window": int(ABC_PENALTY_WINDOW),
            "min_tokens": min(int(ABC_MIN_TOKENS), abc_max),
            "max_tokens": abc_max,
        }

    safe_max = CONTEXT - prefix_len - abc_reserve - 1
    if safe_max < 1:
        raise ValueError(
            f"STYLE／LYRICS／ABC 已占滿上下文（{CONTEXT} tokens）。"
            "請縮短提示詞／樂譜、降低 ABC_MAX_TOKENS，或改用 COT=off。"
        )
    raw_max = int(MUSIC_MAX_TOKENS_OVERRIDE)
    default_max = int(default_sem.max_tokens)
    wanted = raw_max or (round(MAX_DURATION_SEC * 25) if MAX_DURATION_SEC is not None else default_max)
    actual = min(wanted, safe_max)
    if actual < wanted:
        print(f"⚠️ 上下文保護：music token 預算 {wanted} → {actual}（約 {actual / 25:.1f} 秒）")
    music_min = min(int(MUSIC_MIN_TOKENS), actual)
    if music_min != int(MUSIC_MIN_TOKENS):
        print(f"⚠️ Music min_tokens：{MUSIC_MIN_TOKENS} → {music_min}（避免超過 max_tokens）")
    request["semantic_sampling"] = {
        "temperature": float(MUSIC_TEMPERATURE),
        "top_p": float(MUSIC_TOP_P),
        "top_k": int(MUSIC_TOP_K),
        "repetition_penalty": float(MUSIC_REPETITION_PENALTY),
        "penalty_window": int(MUSIC_PENALTY_WINDOW),
        "min_tokens": music_min,
        "max_tokens": int(actual),
    }
    if raw_max:
        print(f"⏱️ Music 上限 {raw_max} tokens（實際預算 {actual}，約 {actual / 25:.1f} 秒）")
    elif MAX_DURATION_SEC is not None:
        print(f"⏱️ 歌曲最長 {MAX_DURATION_SEC} 秒；實際預算 {actual} tokens（約 {actual / 25:.1f} 秒）")
    else:
        print(f"⏱️ 官方預設時間上限 {default_max} tokens（實際 {actual}，約 {actual / 25:.1f} 秒）")
    if ABC_OVERRIDE_TEXT is not None:
        print(f"🎼 使用手動 ABC（{len(ABC_OVERRIDE_TEXT)} 字元）")
    return request


def audio_duration(path: Path):
    info = sf.info(str(path))
    return info.frames / info.samplerate


def make_mp3(flac_path: Path, mp3_path: Path):
    r = subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(flac_path),
                        "-codec:a", "libmp3lame", "-b:a", MP3_BITRATE, str(mp3_path)],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0, r.stdout


DEMUCS_ENV = Path("/content/demucs_env")
STEM_ROOT = Path("/content/yue2_vocal_stems")
LAST_LRC_INFO = {}      # 最近一次 run_lrc 的結果：對上幾行、有沒有用人聲檔


def ensure_demucs():
    """Demucs 裝在獨立環境，不會動到 YuE2 鎖定的 torch 等套件版本。第一次用到才安裝。"""
    py = DEMUCS_ENV / "bin" / "python"
    if py.exists() and (DEMUCS_ENV / ".ready").exists():
        return py
    print("⏳ 第一次分離人聲：安裝 Demucs（獨立環境，約 1 分鐘）……")
    steps = [[sys.executable, "-m", "pip", "install", "-q", "uv"],
             [sys.executable, "-m", "uv", "venv", str(DEMUCS_ENV), "--python", "3.11", "-q"],
             [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", str(py),
              "torch==2.5.1", "torchaudio==2.5.1", "demucs==4.0.1", "soundfile"]]
    for cmd in steps:
        r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            raise RuntimeError(f"Demucs 安裝失敗：{(r.stdout or '')[-400:]}")
    (DEMUCS_ENV / ".ready").write_text("ok")
    return py


def separate_vocals(audio_path):
    """用 Demucs 分離出人聲，回傳人聲檔路徑。先試 GPU，失敗（例如顯存不足）再用 CPU。"""
    audio_path = Path(audio_path)
    out = STEM_ROOT / hashlib.md5(str(audio_path).encode("utf-8")).hexdigest()[:12]
    stem = out / "htdemucs" / "audio" / "vocals.wav"
    if stem.exists():
        return stem
    py = ensure_demucs()
    out.mkdir(parents=True, exist_ok=True)
    local = out / f"audio{audio_path.suffix}"      # 複製到本機：路徑不含中文，也比直接讀雲端硬碟快
    shutil.copy2(audio_path, local)
    log = ""
    for device in (("cuda", "cpu") if torch.cuda.is_available() else ("cpu",)):
        r = subprocess.run([str(py), "-m", "demucs", "-n", "htdemucs", "--two-stems", "vocals", "-d", device,
                            "-o", str(out), str(local)], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        log = r.stdout or ""
        if r.returncode == 0 and stem.exists():
            return stem
    raise RuntimeError(f"Demucs 沒有產生人聲檔：{log[-300:]}")


def run_lrc(audio_path, lyrics_path, lrc_path, segments_path):
    """產生 LRC。回傳 (是否至少對上一行, 紀錄文字)；實際對上幾行記在 LAST_LRC_INFO。"""
    global LAST_LRC_INFO
    notes, align_audio = [], audio_path
    # 直接拿完整混音去辨識，Whisper 常把整段合併成少數幾段，結果一行都對不上；先分離人聲準確很多。
    if globals().get("LRC_SEPARATE_VOCALS", True):
        try:
            align_audio = separate_vocals(audio_path)
            notes.append(f"vocal stem (Demucs): {align_audio}")
        except Exception as e:
            notes.append(f"⚠️ 人聲分離失敗，改用完整混音對歌詞：{e}")
            print(notes[-1])

    def build(device):
        cmd = ["lyric-align", str(align_audio), str(lyrics_path), "--language", LRC_LANGUAGE,
               "--model", LRC_WHISPER_MODEL, "--device", device,
               "--dump-segments", str(segments_path), "-o", str(lrc_path)]
        if LRC_NO_VAD:
            cmd.append("--no-vad")
        if LRC_INTERPOLATE:
            cmd.append("--interpolate")
        return cmd

    def execute(device):
        try:
            return subprocess.run(build(device), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        except FileNotFoundError as e:
            return subprocess.CompletedProcess(build(device), 127, stdout=f"找不到 lyric-align：{e}（請重新執行 ①）")

    r = execute(LRC_DEVICE)
    if r.returncode != 0 and LRC_DEVICE == "cuda":
        if any(m in r.stdout.lower() for m in ["cublas", "cudnn", "cuda error", "cuda runtime", "cannot be loaded", "out of memory"]):
            print("⚠️ GPU 版 Whisper 無法使用，改用 CPU")
            r = execute("cpu")
    found = re.search(r"aligned (\d+)/(\d+) lines", r.stdout or "")
    matched, total = (int(found.group(1)), int(found.group(2))) if found else (None, None)
    LAST_LRC_INFO = {"matched": matched, "total": total, "separated": align_audio is not audio_path}
    # 指令成功但一行都沒對上，不算成功。
    return r.returncode == 0 and matched != 0, "\n".join(notes + [r.stdout or ""])


def save_params_txt(path, song_name, seed, request, duration_sec, truncated, lrc_ok, mp3_ok):
    path.write_text(f"""歌曲：{song_name}
生成時間：{datetime.now().isoformat(timespec="seconds")}

模型：{MODEL_ID}｜COT：{COT_MODE}｜ABC_SOURCE：{ABC_SOURCE}｜CFG：{CFG_SCALE}｜MAX_DURATION_SEC：{MAX_DURATION_SEC}｜ODE_STEPS：{ODE_STEPS}
Seed：{seed}（{SEED_MODE}）｜MUSIC_MAX_TOKENS_OVERRIDE：{MUSIC_MAX_TOKENS_OVERRIDE}
長度：{duration_sec:.2f} 秒｜truncated：{truncated}
LRC：{"成功" if lrc_ok else "失敗或未啟用"}（{LRC_LANGUAGE}，{LRC_WHISPER_MODEL}，no-VAD {LRC_NO_VAD}，interpolate {LRC_INTERPOLATE}，人聲分離 {globals().get('LRC_SEPARATE_VOCALS', True)}）
MP3：{"成功" if mp3_ok else "未輸出"}（{MP3_BITRATE}）

===== STYLE =====
{STYLE}

===== LYRICS =====
{LYRICS}

===== REQUEST =====
{json.dumps(request, ensure_ascii=False, indent=2)}
""", encoding="utf-8")


print("✅ 工具就緒｜專案資料夾：", PROJECT_DIR)


In [ ]:
# @title ④ 🚀 載入 YuE2
# 同一個 Runtime 只需載入一次。主模型採延遲載入，第一次生成時才會放上 GPU。
try:
    import transformers
    from transformers import GenerationMixin, PretrainedConfig, PreTrainedModel
except Exception as e:
    raise RuntimeError("Transformers 狀態異常，請重新執行 ①（若提示需要重啟，重啟後再執行一次）。") from e

YUE2_BACKUP_NAME = "yue2_pipeline_" + re.sub(r"[^\w.-]+", "_", MODEL_ID)
YUE2_LOCAL_BACKUP = Path("/content/yue2_backup_local")

if "pipe" in globals() and pipe is not None:
    print("✅ YuE2 已在記憶體中，略過")
else:
    YUE2_FROM_BACKUP = False
    try:
        print("載入", MODEL_ID)
        pipe = YuE2Pipeline.from_pretrained(MODEL_ID, device="cuda")
    except Exception as e:
        print(f"⚠️ 從 Hugging Face 載入失敗：{e!r}\n→ 改用 Drive 備份")
        if not restore_dirs(YUE2_BACKUP_NAME, YUE2_LOCAL_BACKUP):
            raise RuntimeError("無法取得 YuE2 模型（Hugging Face 與 Drive 備份都不可用）。") from e
        pipe = YuE2Pipeline.from_pretrained(str(YUE2_LOCAL_BACKUP / "pipeline"), device="cuda")
        YUE2_FROM_BACKUP = True
    print("✅ Pipeline 建立完成" + ("（來自 Drive 備份）" if YUE2_FROM_BACKUP else ""))


## ⑤G 圖形介面（選用）

想用圖形介面就保持 `USE_GUI` 勾選（必須先執行 ④）。勾選時，下面的 ⑤ 表單批量生成會自動略過；想用原本的表單流程就取消勾選。

- **風格、歌詞、樂譜都是文字框**，直接貼上就能用，不用改程式碼。面板開啟時會帶入 ③ 各格目前的設定值。
- **專案名稱和歌曲名稱都能在面板上改**：專案名稱是資料夾，歌曲名稱後面會自動加編號（例如 `song_v01`），可以用中文。
- **先看樂譜再生成**：按「先產生樂譜」只會規劃 ABC 樂譜（很快），可以直接在樂譜欄修改，再按「生成歌曲」。樂譜欄有內容時，本批每一首都會用這份樂譜；留空則每首自動規劃。
- **樂譜欄下方會顯示預估長度**（小節數 × 速度的近似值），超過「最長秒數」會警告。「最長秒數」只是上限，超過就直接截斷；想要短一點的完整歌曲，請縮短歌詞或刪減樂譜小節。被截斷的歌會在狀態欄和歌曲瀏覽標示。
- **停止**：會在下一個檢查點停下，已完成的歌會保留。
- **歌曲瀏覽**：回聽之前的歌、看 LRC 和參數，也可以把某一首的樂譜帶回「生成」分頁修改後重新生成。
- 勾選 `OPEN_IN_NEW_TAB`：這格會印出 `gradio.live` 網址和帳號密碼，在新分頁登入使用。網址是公開的，請不要把帳號密碼分享出去。取消勾選則內嵌在這格下方，不需要密碼。如果拿不到臨時網址，會自動改用內嵌模式。
- **免費版 Colab 請注意**：Colab 規定免費執行階段不能主要透過網頁介面操作，使用圖形介面可能被中斷。付費方案且運算單元有餘額時不受此限。
- 面板開著的期間請保持 Colab 分頁開啟；用完請到「執行階段 → 中斷連線並刪除執行階段」。

> YuE2 鎖定 `huggingface-hub 0.36`，只能搭配 Gradio 5。這格會自動安裝相容的版本。

In [ ]:
# @title ⑤G 🖥️ 圖形介面（Gradio）
# @markdown 勾選後執行這格會開啟圖形介面，下面的 ⑤ 表單批量生成會自動略過。必須先執行 ④。
USE_GUI = True # @param {type:"boolean"}
# @markdown **另開分頁**：產生 `gradio.live` 臨時網址，用這格印出的帳號密碼登入。取消勾選則直接內嵌在這格下方（不需要密碼）。
OPEN_IN_NEW_TAB = True # @param {type:"boolean"}
# @markdown 密碼留空的話，每次執行會自動產生一組新的。
GUI_PASSWORD = "" # @param {type:"string"}

YUE2_GUI_ACTIVE = bool(USE_GUI)


def _yue2_gui_main():
    import csv
    import importlib
    import importlib.metadata
    import json
    import random
    import re
    import secrets
    import shutil
    import subprocess
    import sys
    import threading
    import time
    import traceback
    from datetime import datetime
    from pathlib import Path

    G = globals()
    if G.get("pipe") is None:
        raise RuntimeError("請先執行 ④ 載入 YuE2，再執行 ⑤G。")
    for needed in ("build_request", "validate_sampling_options", "apply_pipeline_settings", "pick_seed",
                   "next_version_number", "audio_duration", "run_lrc", "make_mp3", "save_params_txt"):
        if needed not in G:
            raise RuntimeError("找不到共用工具函式，請先執行「🧰 共用工具函式」那一格。")

    # ── Gradio：YuE2 鎖定 huggingface-hub 0.36，只能搭配 Gradio 5 ──
    def installed(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None

    gradio_ver = installed("gradio")
    if gradio_ver is None or gradio_ver.split(".")[0] != "5":
        hub_ver = installed("huggingface_hub") or "0.36.2"
        print(f"安裝與 YuE2 相容的 Gradio 5（目前：{gradio_ver or '未安裝'}），約一分鐘…", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        "gradio==5.50.0", f"huggingface-hub=={hub_ver}"], check=True)
        importlib.invalidate_caches()
        for name in [m for m in sys.modules if m.split(".")[0] in ("gradio", "gradio_client")]:
            del sys.modules[name]
        if "REQUIRED" in G and "version_ok" in G:
            changed = [p for p in G["REQUIRED"] if not G["version_ok"](p)]
            if changed:
                print("⚠️ 安裝 Gradio 後，這些套件的版本和 YuE2 要求的不同：" + "、".join(changed)
                      + "。如果生成時出錯，請重新執行 ①。")
    import gradio as gr

    output_root = Path(G.get("OUTPUT_ROOT", "/content/drive/MyDrive/AI音樂/YuE2"))
    output_root.mkdir(parents=True, exist_ok=True)
    cancel = threading.Event()
    LRC_EMPTY = ("（LRC 是空的：沒有對上任何一行歌詞。請確認「LRC 歌詞時間軸與輸出」有勾選「先分離人聲再對歌詞」，"
                 "再用 ⑤B 補做；歌曲太短或被截斷時也會這樣。詳見歌曲資料夾裡的 lrc_log.txt）")
    lrc_labels = {}      # 歌名 → 「12/16 行」，給狀態欄顯示實際對上幾行

    def as_int(value, name, default=None):
        if value is None or value == "":
            if default is None:
                raise gr.Error(f"{name} 不能空白。")
            return default
        try:
            return int(round(float(value)))
        except (TypeError, ValueError):
            raise gr.Error(f"{name} 必須是數字。") from None

    def apply_settings(v):
        """把面板上的值寫回 notebook 的設定變數，之後沿用原本表單流程的函式。"""
        cfg_text = str(v["cfg"] or "").strip()
        try:
            cfg = float(cfg_text) if cfg_text else None
        except ValueError:
            raise gr.Error("CFG 必須是數字，或留空使用官方預設。") from None
        if cfg is not None and not 0.0 <= cfg <= 20.0:
            raise gr.Error("CFG 需介於 0～20；留空則使用官方預設。")
        max_sec = as_int(v["max_sec"], "最長秒數", 0)
        if not 0 <= max_sec <= 900:
            raise gr.Error("最長秒數需為 0～900（0 = 官方預設）。")
        seed_list = [int(s) for s in re.split(r"[,\s，、]+", str(v["seed_list"] or "").strip()) if s.strip().lstrip("-").isdigit()]
        batch = as_int(v["batch"], "批量數", 1)
        if v["seed_mode"] == "list" and len(seed_list) < batch:
            raise gr.Error(f"Seed 清單只有 {len(seed_list)} 個，不夠生成 {batch} 首。")
        abc_text = str(v.get("abc") or "").strip()
        if abc_text and v["cot"] == "off":
            raise gr.Error("規劃模式選 off 時不能使用樂譜；請改成 full 或 melody，或清空樂譜。")
        G.update(
            STYLE=str(v["style"] or "").strip(), LYRICS=str(v["lyrics"] or "").strip(),
            COT_MODE=v["cot"], CFG_SCALE=cfg, CFG_SCALE_TEXT=cfg_text,
            MAX_DURATION_SEC=(max_sec if max_sec > 0 else None),
            BATCH_COUNT=batch, SEED_MODE=v["seed_mode"],
            FIXED_SEED=as_int(v["fixed_seed"], "固定 Seed", 123456), SEED_LIST=seed_list,
            ABC_OVERRIDE_TEXT=(abc_text or None), ABC_SOURCE=("gui" if abc_text else "auto"),
            ABC_TEMPERATURE=float(v["abc_temp"]), ABC_TOP_P=float(v["abc_top_p"]),
            ABC_TOP_K=as_int(v["abc_top_k"], "ABC Top K"),
            ABC_REPETITION_PENALTY=float(v["abc_rep"]),
            ABC_PENALTY_WINDOW=as_int(v["abc_window"], "ABC Penalty Window"),
            ABC_MIN_TOKENS=as_int(v["abc_min"], "ABC Min Tokens"),
            ABC_MAX_TOKENS=as_int(v["abc_max"], "ABC Max Tokens"),
            MUSIC_TEMPERATURE=float(v["mu_temp"]), MUSIC_TOP_P=float(v["mu_top_p"]),
            MUSIC_TOP_K=as_int(v["mu_top_k"], "Music Top K"),
            MUSIC_REPETITION_PENALTY=float(v["mu_rep"]),
            MUSIC_PENALTY_WINDOW=as_int(v["mu_window"], "Music Penalty Window"),
            MUSIC_MIN_TOKENS=as_int(v["mu_min"], "Music Min Tokens"),
            MUSIC_MAX_TOKENS_OVERRIDE=as_int(v["mu_max"], "Music Max Tokens", 0),
            ODE_STEPS=as_int(v["ode"], "ODE Steps"),
            MAKE_LRC=bool(v["make_lrc"]), LRC_LANGUAGE=v["lrc_lang"], LRC_NO_VAD=bool(v["lrc_no_vad"]),
            LRC_INTERPOLATE=bool(v["lrc_interp"]), LRC_WHISPER_MODEL=str(v["lrc_model"]).strip() or "medium",
            LRC_SEPARATE_VOCALS=bool(v["lrc_separate"]),
            LRC_DEVICE=v["lrc_device"], MAKE_MP3=bool(v["make_mp3"]), MP3_BITRATE=v["mp3_bitrate"],
        )
        if not G["STYLE"]:
            raise gr.Error("請先填風格（STYLE）。")
        try:
            G["validate_sampling_options"]()
        except (ValueError, TypeError) as e:
            raise gr.Error(str(e)) from None
        G["apply_pipeline_settings"]()

    def make_progress(progress, head, request):
        state = {"phase": None, "n": 0, "last": 0.0}
        abc_max = int((request.get("abc_sampling") or {}).get("max_tokens") or 4096)
        music_max = int((request.get("semantic_sampling") or {}).get("max_tokens") or 9000)

        def on_token(phase, token):
            if phase != state["phase"]:
                state.update(phase=phase, n=0)
            state["n"] += 1
            now = time.time()
            if now - state["last"] < 0.8:
                return
            state["last"] = now
            n = state["n"]
            if phase == "abc":
                progress(min(n / abc_max, 1.0), desc=f"{head}｜規劃樂譜：{n} tokens")
            else:
                progress(min(n / music_max, 1.0),
                         desc=f"{head}｜生成歌曲：約 {n / 25:.0f} 秒（上限 {music_max / 25:.0f} 秒）")

        def on_stage(stage, done, total):
            label = "合成音訊" if stage == "nar" else "解碼音訊"
            progress((done / total) if total else 0.0, desc=f"{head}｜{label}：{done}/{total}")

        return on_token, on_stage

    def read_text(path):
        try:
            return Path(path).read_text(encoding="utf-8-sig")
        except OSError:
            return ""

    def estimate_score(abc_text):
        """從 ABC 樂譜估算歌曲長度：小節數 × 每小節秒數。只看開頭的拍號與速度，不展開反覆記號，所以是近似值。"""
        text = str(abc_text or "")
        m = re.search(r"^M:\s*(\d+)\s*/\s*(\d+)", text, re.M)
        q = re.search(r"^Q:\s*(?:(\d+)\s*/\s*(\d+)\s*=\s*)?(\d+)", text, re.M)
        if not m or not q:
            return None
        beat = (int(q.group(1)) / int(q.group(2))) if q.group(1) else 0.25
        bpm, num, den = int(q.group(3)), int(m.group(1)), int(m.group(2))
        if bpm <= 0 or beat <= 0 or den <= 0:
            return None
        bar_sec = (num / den) / beat * 60 / bpm
        counts, voice = {}, "_"
        for line in text.splitlines():
            v = re.match(r"^V:\s*(\S+)", line)
            if v:
                voice = v.group(1)
                continue
            if "|" not in line or re.match(r"^[A-Za-z]:", line) or line.lstrip().startswith("%"):
                continue
            for seg in line.split("|"):
                seg = seg.strip().strip(":[]").strip()
                if not seg:
                    continue
                z = re.fullmatch(r"Z(\d+)", seg)
                counts[voice] = counts.get(voice, 0) + (int(z.group(1)) if z else 1)
        bars = max(counts.values(), default=0)
        if not bars:
            return None
        return {"bars": bars, "seconds": bars * bar_sec, "meter": f"{num}/{den}", "bpm": bpm}

    def estimate_label(est):
        return f"約 {est['seconds']:.0f} 秒（{est['bars']} 小節，{est['meter']} 拍，{est['bpm']} BPM）"

    def score_note(abc_text, max_sec=None):
        """樂譜欄下方的說明：預估長度，以及會不會超過最長秒數。"""
        if not str(abc_text or "").strip():
            return "樂譜欄是空的：生成時每首歌會自動規劃樂譜。"
        est = estimate_score(abc_text)
        if est is None:
            return "看不出這份樂譜的拍號或速度，無法估算長度。"
        note = f"**預估長度：{estimate_label(est)}**。這是用小節數算的近似值。"
        try:
            limit = int(float(max_sec or 0)) or 360
        except (TypeError, ValueError):
            limit = 360
        if est["seconds"] > limit:
            note += (f"\n\n⚠️ 超過目前的最長秒數（{limit} 秒），生成時會被截斷。"
                     "可以刪掉幾個小節、重新產生樂譜，或調高最長秒數。")
        return note

    def status_text(head, rows, notes=()):
        lines = [head]
        if rows:
            lines += ["", "本次結果："]
            lines += [f"  {r['song']}｜seed {r['seed']}｜{r['duration_sec']} 秒｜LRC {lrc_labels.get(r['song'], '✓' if r['lrc_ok'] else '✗')}｜{r['status']}"
                      for r in rows]
        if notes:
            lines += [""] + list(notes)
        return "\n".join(lines)

    KEYS = ["style", "lyrics", "cot", "batch", "seed_mode", "fixed_seed", "seed_list", "max_sec", "cfg", "abc",
            "abc_temp", "abc_top_p", "abc_top_k", "abc_rep", "abc_window", "abc_min", "abc_max",
            "mu_temp", "mu_top_p", "mu_top_k", "mu_rep", "mu_window", "mu_min", "mu_max", "ode",
            "make_lrc", "lrc_lang", "lrc_no_vad", "lrc_interp", "lrc_model", "lrc_device", "lrc_separate",
            "make_mp3", "mp3_bitrate"]

    def do_plan(*values, progress=gr.Progress()):
        v = dict(zip(KEYS, values))
        if v["cot"] == "off":
            raise gr.Error("規劃模式選 off 時不會產生樂譜；請改成 full 或 melody。")
        v["abc"] = ""
        apply_settings(v)
        cancel.clear()
        seed = int(G["FIXED_SEED"]) if v["seed_mode"] in ("fixed", "sequence") else random.SystemRandom().randint(1, 2_147_483_647)
        try:
            request = G["build_request"]("plan", seed)
        except (ValueError, TypeError) as e:
            raise gr.Error(str(e)) from None
        on_token, _ = make_progress(progress, "先產生樂譜", request)
        kwargs = {k: request[k] for k in ("style", "lyrics", "cot", "seed", "id", "cfg_scale") if k in request}
        t0 = time.time()
        try:
            plan = G["pipe"].plan(**kwargs, abc_sampling=request.get("abc_sampling"),
                                  cancelled=cancel.is_set, on_token=on_token)
        except InterruptedError:
            return gr.update(), "⏹️ 已停止，樂譜沒有變更。"
        except Exception as e:
            traceback.print_exc()
            raise gr.Error(f"產生樂譜失敗：{e!r}"[:600]) from None
        note = "（已達 token 上限，樂譜可能不完整）" if plan.truncated else ""
        est = estimate_score(plan.abc)
        length = f"預估長度：{estimate_label(est)}。\n" if est else ""
        return plan.abc or "", (f"✅ 樂譜已產生，seed {seed}，耗時 {time.time() - t0:.0f} 秒{note}。\n{length}"
                                "可以直接修改樂譜，再按「生成歌曲」。樂譜欄有內容時，本批每一首都會用這份樂譜。")

    def do_generate(project, name_prefix, *values, progress=gr.Progress()):
        v = dict(zip(KEYS, values))
        apply_settings(v)
        cancel.clear()
        project = re.sub(r'[\\/:*?"<>|]+', "_", str(project or "").strip()) or "我的歌曲"
        project_dir = output_root / project
        project_dir.mkdir(parents=True, exist_ok=True)
        prefix = re.sub(r'[\\/:*?"<>|\s]+', "_", str(name_prefix or "").strip()) or str(G.get("FILE_PREFIX", "song_v"))
        digits = int(G.get("VERSION_DIGITS", 2))
        batch = int(G["BATCH_COUNT"])
        start = G["next_version_number"](project_dir, prefix)
        rows, notes, last_audio, last_lrc, stopped = [], [], None, "", False
        for i in range(batch):
            song_name = f"{prefix}{start + i:0{digits}d}"
            output_dir = project_dir / song_name
            if output_dir.exists():
                raise gr.Error(f"{output_dir} 已存在，停止以免覆蓋。")
            seed = G["pick_seed"](i)
            head = f"[{i + 1}/{batch}] {song_name}（seed {seed}）"
            # YuE2 的 request id 只接受英數字；歌曲名稱有中文時，id 改用英數代號，檔名仍用你取的名稱。
            song_id = song_name if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,179}", song_name) else f"song_{start + i:0{digits}d}"
            try:
                request = G["build_request"](song_id, seed)
            except (ValueError, TypeError) as e:
                raise gr.Error(str(e)) from None
            yield status_text(f"🎵 {head} 生成中…", rows, notes), last_audio, last_lrc
            on_token, on_stage = make_progress(progress, head, request)
            try:
                song = G["pipe"](**request, cancelled=cancel.is_set, on_token=on_token, on_progress=on_stage)
                song.save_artifacts(str(output_dir))
                original = output_dir / "audio.flac"
                if not original.exists():
                    raise FileNotFoundError("YuE2 完成但找不到 audio.flac")
                named_flac = output_dir / f"{song_name}.flac"
                shutil.copy2(original, named_flac)
                lyrics_path = output_dir / f"{song_name}_lyrics.txt"
                lyrics_path.write_text(G["LYRICS"], encoding="utf-8")
                (output_dir / "notebook_request.json").write_text(
                    json.dumps(request, ensure_ascii=False, indent=2), encoding="utf-8")
                duration = G["audio_duration"](named_flac)
                truncated = getattr(song, "truncated", None)
                lrc_ok = False
                if G["MAKE_LRC"] and G["LYRICS"]:
                    progress(1.0, desc=f"{head}｜產生 LRC 歌詞時間軸…")
                    lrc_ok, lrc_log = G["run_lrc"](named_flac, lyrics_path, output_dir / f"{song_name}.lrc",
                                                   output_dir / f"{song_name}_segments.json")
                    (output_dir / "lrc_log.txt").write_text(lrc_log, encoding="utf-8")
                    info = G.get("LAST_LRC_INFO") or {}
                    if info.get("total"):
                        lrc_labels[song_name] = f"{info['matched']}/{info['total']} 行"
                mp3_ok = False
                if G["MAKE_MP3"]:
                    progress(1.0, desc=f"{head}｜轉 MP3…")
                    mp3_ok, mp3_log = G["make_mp3"](named_flac, output_dir / f"{song_name}.mp3")
                    if not mp3_ok:
                        (output_dir / "mp3_log.txt").write_text(mp3_log, encoding="utf-8")
                G["save_params_txt"](output_dir / "參數.txt", song_name, seed, request, duration, truncated, lrc_ok, mp3_ok)
                rows.append({"song": song_name, "seed": seed, "duration_sec": round(duration, 2),
                             "truncated": str(truncated), "lrc_ok": lrc_ok, "mp3_ok": mp3_ok,
                             "folder": str(output_dir), "status": "ok"})
                # 被長度上限截斷時講清楚：樂譜原本規劃多長、實際切在哪裡。
                if isinstance(truncated, dict) and truncated.get("semantic"):
                    est = estimate_score(read_text(output_dir / "score.abc"))
                    planned = f"樂譜規劃{estimate_label(est)}，" if est else ""
                    notes.append(f"⚠️ {song_name} 被長度上限截斷：{planned}實際只生成 {duration:.0f} 秒。")
                if isinstance(truncated, dict) and truncated.get("abc"):
                    notes.append(f"⚠️ {song_name} 的樂譜達到 token 上限，可能不完整。")
                mp3 = output_dir / f"{song_name}.mp3"
                last_audio = str(mp3 if mp3.exists() else named_flac)
                if lrc_ok:
                    last_lrc = read_text(output_dir / f"{song_name}.lrc").strip() or LRC_EMPTY
                else:
                    last_lrc = ""
                    if G["MAKE_LRC"]:
                        empty = (G.get("LAST_LRC_INFO") or {}).get("matched") == 0
                        last_lrc = LRC_EMPTY if empty else "（LRC 產生失敗，詳見歌曲資料夾裡的 lrc_log.txt）"
                yield status_text(f"✅ {head} 完成，{duration:.1f} 秒。", rows, notes), last_audio, last_lrc
            except InterruptedError:
                stopped = True
                break
            except Exception as e:
                traceback.print_exc()
                fail_dir = project_dir / f"{song_name}_FAILED"
                fail_dir.mkdir(parents=True, exist_ok=True)
                (fail_dir / "ERROR.txt").write_text(traceback.format_exc(), encoding="utf-8")
                (fail_dir / "request.json").write_text(json.dumps(request, ensure_ascii=False, indent=2), encoding="utf-8")
                rows.append({"song": song_name, "seed": seed, "duration_sec": "", "truncated": "",
                             "lrc_ok": False, "mp3_ok": False, "folder": str(fail_dir), "status": f"失敗：{e!r}"[:120]})
                yield status_text(f"❌ {head} 失敗，繼續下一首。", rows, notes), last_audio, last_lrc
        if rows:
            summary = project_dir / f"batch_{datetime.now():%Y%m%d_%H%M%S}.csv"
            with summary.open("w", newline="", encoding="utf-8-sig") as f:
                w = csv.DictWriter(f, fieldnames=["song", "seed", "duration_sec", "truncated", "lrc_ok", "mp3_ok", "folder", "status"])
                w.writeheader()
                w.writerows(rows)
        ok = sum(1 for r in rows if r["status"] == "ok")
        head = (f"⏹️ 已停止。完成 {ok} 首。" if stopped else f"🎉 全部完成：成功 {ok} 首，失敗 {len(rows) - ok} 首。")
        if any("截斷" in n for n in notes):
            notes = notes + ["想要完整的歌：縮短歌詞、先產生樂譜後刪掉幾個小節，或調高最長秒數。"]
        yield status_text(head + f"\n資料夾：{project_dir}", rows, notes), last_audio, last_lrc

    def do_stop():
        cancel.set()
        return "⏹️ 已送出停止要求，會在目前這個步驟的下一個檢查點停下…"

    # ── 歌曲瀏覽 ──
    def list_songs():
        found = []
        if output_root.is_dir():
            for proj in output_root.iterdir():
                if not proj.is_dir():
                    continue
                for song in proj.iterdir():
                    if song.is_dir() and ((song / f"{song.name}.flac").exists() or (song / "audio.flac").exists()):
                        found.append((song.stat().st_mtime, f"{proj.name}/{song.name}"))
        return [name for _, name in sorted(found, reverse=True)[:300]]

    def show_song(rel):
        if not rel:
            return None, "", "", "", ""
        folder = (output_root / rel).resolve()
        if output_root.resolve() not in folder.parents or not folder.is_dir():
            return None, "", "找不到這個資料夾。", "", ""
        name = folder.name
        audio = next((p for p in (folder / f"{name}.mp3", folder / f"{name}.flac", folder / "audio.flac") if p.exists()), None)
        lrc_file = folder / f"{name}.lrc"
        lrc_text = (read_text(lrc_file).strip() or LRC_EMPTY) if lrc_file.exists() else "（這首沒有 LRC）"
        score = read_text(folder / "score.abc")
        est = estimate_score(score)
        length = f"**樂譜預估長度：{estimate_label(est)}**" if est else ""
        if est and audio:
            try:
                actual = G["audio_duration"](folder / f"{name}.flac" if (folder / f"{name}.flac").exists() else folder / "audio.flac")
                length += f"｜實際音訊 {actual:.0f} 秒"
                if actual < est["seconds"] - 5:
                    length += "　⚠️ 比樂譜短，很可能是被最長秒數截斷"
            except Exception:
                pass
        return (str(audio) if audio else None, lrc_text,
                read_text(folder / "參數.txt") or "（這首沒有參數紀錄）", score, length)

    def refresh_songs():
        items = list_songs()
        first = items[0] if items else None
        return (gr.update(choices=items, value=first),) + show_song(first)

    def use_score(score):
        if not str(score or "").strip():
            raise gr.Error("這首沒有樂譜（規劃模式為 off 的歌不會有）。")
        gr.Info("樂譜已帶到「生成」分頁的樂譜欄。")
        return score

    d = lambda name, default: G.get(name, default)
    cfg_default = d("CFG_SCALE_TEXT", "") if isinstance(d("CFG_SCALE_TEXT", ""), str) else ""
    with gr.Blocks(title="YuE2 歌曲生成") as demo:
        gr.Markdown(f"## 🎵 YuE2 歌曲生成\n模型：`{d('MODEL_ID', 'm-a-p/YuE2-3B')}`｜歌曲存到 `{output_root}`")
        with gr.Tab("生成"):
            with gr.Row():
                with gr.Column(scale=5):
                    with gr.Row():
                        project = gr.Textbox(value=d("PROJECT_NAME", "我的歌曲"), label="專案名稱（歌曲會放在這個資料夾）")
                        name_prefix = gr.Textbox(value=d("FILE_PREFIX", "song_v"),
                                                 label="歌曲名稱（後面會自動加編號，例如 song_v01；可以用中文）")
                    style = gr.Textbox(value=d("STYLE", ""), lines=7, label="風格 STYLE（語言、曲風、樂器、人聲、速度…用逗號或換行分隔）")
                    lyrics = gr.Textbox(value=d("LYRICS", ""), lines=14, label="歌詞 LYRICS（[Verse]、[Chorus] 等段落標籤可保留；留空可嘗試純音樂）")
                    with gr.Row():
                        cot = gr.Radio(["full", "melody", "off"], value=d("COT_MODE", "full"),
                                       label="規劃模式（full 旋律＋和弦｜melody 只有旋律｜off 不規劃）")
                        batch = gr.Slider(1, 20, value=int(d("BATCH_COUNT", 1)), step=1, label="批量數")
                    with gr.Row():
                        seed_mode = gr.Dropdown(["random", "fixed", "sequence", "list"], value=d("SEED_MODE", "random"),
                                                label="Seed 模式")
                        fixed_seed = gr.Number(value=int(d("FIXED_SEED", 123456)), precision=0, label="固定 Seed（fixed／sequence 用）")
                        seed_list = gr.Textbox(value=d("SEED_LIST_TEXT", ""), label="Seed 清單（list 用，逗號分隔）")
                    with gr.Row():
                        max_sec = gr.Number(value=int(d("MAX_DURATION_SEC", 0) or 0), precision=0,
                                            label="最長秒數（0 = 官方預設約 360 秒；上限 900）")
                        cfg = gr.Textbox(value=cfg_default, label="CFG（留空 = 官方預設；可填 0～20）")
                    with gr.Accordion("進階採樣（不確定就保持預設）", open=False):
                        gr.Markdown("**樂譜規劃階段（ABC）**")
                        with gr.Row():
                            abc_temp = gr.Number(value=float(d("ABC_TEMPERATURE", 0.7)), label="Temperature")
                            abc_top_p = gr.Number(value=float(d("ABC_TOP_P", 0.9)), label="Top P")
                            abc_top_k = gr.Number(value=int(d("ABC_TOP_K", 30)), precision=0, label="Top K")
                            abc_rep = gr.Number(value=float(d("ABC_REPETITION_PENALTY", 1.005)), label="Repetition Penalty")
                        with gr.Row():
                            abc_window = gr.Number(value=int(d("ABC_PENALTY_WINDOW", 100)), precision=0, label="Penalty Window（1～100）")
                            abc_min = gr.Number(value=int(d("ABC_MIN_TOKENS", 32)), precision=0, label="Min Tokens")
                            abc_max = gr.Number(value=int(d("ABC_MAX_TOKENS", 4096)), precision=0, label="Max Tokens")
                        gr.Markdown("**歌曲生成階段（Music）**")
                        with gr.Row():
                            mu_temp = gr.Number(value=float(d("MUSIC_TEMPERATURE", 1.0)), label="Temperature")
                            mu_top_p = gr.Number(value=float(d("MUSIC_TOP_P", 0.95)), label="Top P")
                            mu_top_k = gr.Number(value=int(d("MUSIC_TOP_K", 100)), precision=0, label="Top K")
                            mu_rep = gr.Number(value=float(d("MUSIC_REPETITION_PENALTY", 1.2)), label="Repetition Penalty")
                        with gr.Row():
                            mu_window = gr.Number(value=int(d("MUSIC_PENALTY_WINDOW", 50)), precision=0, label="Penalty Window（1～100）")
                            mu_min = gr.Number(value=int(d("MUSIC_MIN_TOKENS", 200)), precision=0, label="Min Tokens")
                            mu_max = gr.Number(value=int(d("MUSIC_MAX_TOKENS_OVERRIDE", 0)), precision=0,
                                               label="Max Tokens（0 = 依最長秒數；和最長秒數只能擇一）")
                            ode = gr.Number(value=int(d("ODE_STEPS", 32)), precision=0, label="ODE Steps（官方預設 32）")
                    with gr.Accordion("LRC 歌詞時間軸與輸出", open=False):
                        with gr.Row():
                            make_lrc = gr.Checkbox(value=bool(d("MAKE_LRC", True)), label="產生 LRC")
                            lrc_lang = gr.Dropdown(["zh", "en", "ja"], value=d("LRC_LANGUAGE", "en"), label="歌詞語言")
                            lrc_model = gr.Dropdown(["small", "medium", "large-v3"], value=d("LRC_WHISPER_MODEL", "medium"),
                                                    label="Whisper 模型", allow_custom_value=True)
                            lrc_device = gr.Dropdown(["cpu", "cuda"], value=d("LRC_DEVICE", "cpu"), label="LRC 裝置")
                        with gr.Row():
                            lrc_no_vad = gr.Checkbox(value=bool(d("LRC_NO_VAD", True)), label="不使用 VAD（慢歌、長音多建議勾選）")
                            lrc_interp = gr.Checkbox(value=bool(d("LRC_INTERPOLATE", False)), label="對不上的行用推測時間補齊")
                            lrc_separate = gr.Checkbox(value=bool(d("LRC_SEPARATE_VOCALS", True)),
                                                       label="先分離人聲再對歌詞（準確很多；第一次多花約 1 分鐘）")
                            make_mp3 = gr.Checkbox(value=bool(d("MAKE_MP3", True)), label="另存 MP3（FLAC 母檔一定保留）")
                            mp3_bitrate = gr.Dropdown(["192k", "256k", "320k"], value=d("MP3_BITRATE", "320k"), label="MP3 位元率")
                    with gr.Row():
                        go = gr.Button("生成歌曲", variant="primary", scale=3)
                        stop = gr.Button("停止", variant="stop", scale=1)
                with gr.Column(scale=4):
                    status = gr.Textbox(label="狀態", lines=8, interactive=False)
                    audio = gr.Audio(label="最新歌曲", type="filepath", interactive=False)
                    lrc_out = gr.Textbox(label="LRC 歌詞時間軸", lines=8, interactive=False)
                    gr.Markdown("### 樂譜（ABC）\n留空 = 每首歌自動規劃。想先看或修改樂譜，就按「先產生樂譜」，改好再按「生成歌曲」。"
                                "樂譜欄有內容時，本批每一首都會用這份樂譜。")
                    abc = gr.Textbox(label="樂譜（可編輯）", lines=12, value="")
                    abc_note = gr.Markdown(score_note("", None))
                    with gr.Row():
                        plan_btn = gr.Button("先產生樂譜（不生成歌曲）")
                        clear_abc = gr.Button("清空樂譜")
        with gr.Tab("歌曲瀏覽"):
            with gr.Row():
                songs = gr.Dropdown([], value=None, label="歌曲（新的在前）", scale=5)
                songs_refresh = gr.Button("重新整理", scale=1)
            with gr.Row():
                with gr.Column():
                    h_audio = gr.Audio(label="播放", type="filepath", interactive=False)
                    h_lrc = gr.Textbox(label="LRC 歌詞時間軸", lines=10, interactive=False)
                with gr.Column():
                    h_params = gr.Textbox(label="生成參數", lines=10, interactive=False)
                    h_score = gr.Textbox(label="樂譜 score.abc", lines=8, interactive=False)
                    h_length = gr.Markdown("")
                    h_use = gr.Button("把這份樂譜帶到「生成」分頁")

        inputs = [style, lyrics, cot, batch, seed_mode, fixed_seed, seed_list, max_sec, cfg, abc,
                  abc_temp, abc_top_p, abc_top_k, abc_rep, abc_window, abc_min, abc_max,
                  mu_temp, mu_top_p, mu_top_k, mu_rep, mu_window, mu_min, mu_max, ode,
                  make_lrc, lrc_lang, lrc_no_vad, lrc_interp, lrc_model, lrc_device, lrc_separate,
                  make_mp3, mp3_bitrate]
        history_out = [songs, h_audio, h_lrc, h_params, h_score, h_length]
        abc.change(score_note, [abc, max_sec], abc_note, queue=False)
        max_sec.change(score_note, [abc, max_sec], abc_note, queue=False)
        go.click(do_generate, [project, name_prefix] + inputs, [status, audio, lrc_out],
                 concurrency_id="yue2", concurrency_limit=1).then(refresh_songs, None, history_out)
        plan_btn.click(do_plan, inputs, [abc, status], concurrency_id="yue2", concurrency_limit=1)
        stop.click(do_stop, None, status, queue=False)
        clear_abc.click(lambda: "", None, abc, queue=False)
        songs_refresh.click(refresh_songs, None, history_out)
        songs.input(show_song, songs, [h_audio, h_lrc, h_params, h_score, h_length])
        h_use.click(use_score, h_score, abc)
        demo.load(refresh_songs, None, history_out)

    # 重新執行這格時，先把上一次開的介面關掉。
    old = G.get("YUE2_GUI_DEMO")
    if old is not None:
        try:
            old.close()
        except Exception:
            pass
    G["YUE2_GUI_DEMO"] = demo
    demo.queue()
    kw = dict(prevent_thread_lock=True, show_error=True, allowed_paths=[str(output_root)])
    kw.update(G.get("_YUE2_GUI_LAUNCH_OVERRIDES", {}))
    url = None
    if OPEN_IN_NEW_TAB:
        user, pw = "yue2", (GUI_PASSWORD.strip() or secrets.token_urlsafe(9))
        for attempt in range(2):
            try:
                _, _, url = demo.launch(share=True, auth=(user, pw), inline=False, quiet=True, **kw)
            except Exception as e:
                url = None
                print(f"（建立臨時網址時發生錯誤：{e}）")
            if url:
                break
            try:
                demo.close()      # 沒拿到網址：先關掉，重試或改用內嵌模式
            except Exception:
                pass
            if attempt == 0:
                print(" 第一次沒有拿到臨時網址，再試一次……")
        print("=" * 72)
        if url:
            print(f" 圖形介面網址：{url}\n 帳號：{user}\n 密碼：{pw}")
            print(" 這個網址是公開的，只在這次執行階段有效；請不要分享帳號密碼。")
        else:
            print(" 沒有成功產生臨時網址，自動改用內嵌模式：面板會直接顯示在這格下方，不需要密碼。")
        print("=" * 72)
    if not url:
        # 內嵌模式：告訴 Gradio 它在 Colab 代理後面的真正網址，音訊才載得出來。
        try:
            import socket
            from google.colab.output import eval_js
            port = 7860
            while port < 7900:
                with socket.socket() as s:
                    if s.connect_ex(("127.0.0.1", port)) != 0:
                        break
                port += 1
            root = str(eval_js(f"google.colab.kernel.proxyPort({port})")).rstrip("/")
            if root.startswith("http"):
                kw.setdefault("server_port", port)
                kw.setdefault("root_path", root)
        except Exception as e:
            print(f"（無法取得 Colab 代理網址，音訊預覽可能載不出來：{e}）")
        demo.launch(share=False, inline=True, height=1400, **kw)
    print("介面啟動後這格就算執行完了，介面會在背景持續運作。Colab 分頁請保持開啟；用完記得中斷執行階段。")


if USE_GUI:
    _yue2_gui_main()
else:
    print("⏭️ 未勾選 USE_GUI，請使用下面的 ⑤ 表單批量生成。")

## ⑤ 生成

每首歌一個資料夾：FLAC（母檔）、MP3、LRC、歌詞、參數、Whisper segments，以及 YuE2 原生輸出（樂譜 `score.abc` 等）。批次結束會另存一份摘要 CSV。

In [ ]:
# @title ⑤ 🎲 批量生成 + 自動 LRC
if globals().get("YUE2_GUI_ACTIVE"):
    print("⏭️ 已開啟圖形介面（⑤G），略過表單批量生成。想用表單批量，請取消勾選 ⑤G 的 USE_GUI，重新執行 ⑤G 再執行這格。")
else:
    import csv
    import shutil

    if "pipe" not in globals() or pipe is None:
        raise RuntimeError("請先執行 ④ 載入 YuE2。")

    validate_sampling_options()
    ABC_OVERRIDE_TEXT = resolve_manual_abc()
    apply_pipeline_settings()

    start_version = next_version_number(PROJECT_DIR, FILE_PREFIX)
    batch_rows, generated_dirs = [], []
    print(f"生成 {BATCH_COUNT} 首，從 {FILE_PREFIX}{start_version:0{VERSION_DIGITS}d} 開始 → {PROJECT_DIR}")

    for i in range(BATCH_COUNT):
        song_name = f"{FILE_PREFIX}{start_version + i:0{VERSION_DIGITS}d}"
        output_dir = PROJECT_DIR / song_name
        if output_dir.exists():
            raise RuntimeError(f"{output_dir} 已存在，停止以免覆蓋。")

        seed = pick_seed(i)
        request = build_request(song_name, seed)
        print("\n" + "=" * 70 + f"\n🎵 [{i + 1}/{BATCH_COUNT}] {song_name}｜seed {seed}")

        try:
            song = pipe(**request)
            song.save_artifacts(str(output_dir))
            generated_dirs.append(output_dir)

            original = output_dir / "audio.flac"
            if not original.exists():
                raise FileNotFoundError("YuE2 完成但找不到 audio.flac")
            named_flac = output_dir / f"{song_name}.flac"
            shutil.copy2(original, named_flac)

            lyrics_path = output_dir / f"{song_name}_lyrics.txt"
            lyrics_path.write_text(LYRICS, encoding="utf-8")
            (output_dir / "notebook_request.json").write_text(json.dumps(request, ensure_ascii=False, indent=2), encoding="utf-8")

            duration = audio_duration(named_flac)
            truncated = getattr(song, "truncated", None)
            print(f"⏱️ 長度 {duration:.2f} 秒｜truncated：{truncated}")

            lrc_ok = False
            if MAKE_LRC:
                print("📝 LRC……")
                lrc_ok, lrc_log = run_lrc(named_flac, lyrics_path, output_dir / f"{song_name}.lrc",
                                          output_dir / f"{song_name}_segments.json")
                (output_dir / "lrc_log.txt").write_text(lrc_log, encoding="utf-8")
                print(lrc_log)
                _lrc = globals().get("LAST_LRC_INFO") or {}
                if _lrc.get("total"):
                    print(f"{'✅' if lrc_ok else '⚠️'} LRC 對上 {_lrc['matched']}/{_lrc['total']} 行"
                          + ("" if lrc_ok else "（歌曲已保存，詳見 lrc_log.txt）"))
                else:
                    print("✅ LRC 完成" if lrc_ok else "⚠️ LRC 失敗（歌曲已保存，詳見 lrc_log.txt）")

            mp3_ok = False
            if MAKE_MP3:
                mp3_ok, mp3_log = make_mp3(named_flac, output_dir / f"{song_name}.mp3")
                if not mp3_ok:
                    (output_dir / "mp3_log.txt").write_text(mp3_log, encoding="utf-8")
                    print("⚠️ MP3 轉檔失敗")

            save_params_txt(output_dir / "參數.txt", song_name, seed, request, duration, truncated, lrc_ok, mp3_ok)
            batch_rows.append({"song": song_name, "seed": seed, "duration_sec": round(duration, 2),
                               "truncated": str(truncated), "lrc_ok": lrc_ok, "mp3_ok": mp3_ok,
                               "folder": str(output_dir), "status": "ok"})
        except Exception as e:
            print("❌ 失敗：", repr(e))
            traceback.print_exc()
            fail_dir = PROJECT_DIR / f"{song_name}_FAILED"
            fail_dir.mkdir(parents=True, exist_ok=True)
            (fail_dir / "ERROR.txt").write_text(traceback.format_exc(), encoding="utf-8")
            (fail_dir / "request.json").write_text(json.dumps(request, ensure_ascii=False, indent=2), encoding="utf-8")
            batch_rows.append({"song": song_name, "seed": seed, "duration_sec": "", "truncated": "",
                               "lrc_ok": False, "mp3_ok": False, "folder": str(fail_dir), "status": "failed"})

    summary = PROJECT_DIR / f"batch_{datetime.now():%Y%m%d_%H%M%S}.csv"
    with summary.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["song", "seed", "duration_sec", "truncated", "lrc_ok", "mp3_ok", "folder", "status"])
        w.writeheader()
        w.writerows(batch_rows)

    print("\n" + "=" * 70 + "\n🎉 完成｜摘要：", summary)
    for r in batch_rows:
        print(f"- {r['song']} | seed={r['seed']} | {r['duration_sec']} 秒 | LRC={r['lrc_ok']} | {r['status']}")


In [ ]:
# @title ⑤B 🛠️ 補做 LRC（勾選才執行，不會重新生成歌曲）
RUN_LRC_REPAIR = False  # @param {type:"boolean"}

if not RUN_LRC_REPAIR:
    print("⏭️ 未勾選，略過")
elif not MAKE_LRC:
    print("③ 的 MAKE_LRC 未勾選")
else:
    pattern = re.compile(rf"^{re.escape(FILE_PREFIX)}\d+$")
    for folder in [p for p in sorted(PROJECT_DIR.iterdir()) if p.is_dir() and pattern.match(p.name)]:
        name = folder.name
        if (folder / f"{name}.lrc").exists():
            continue
        audio = folder / f"{name}.flac"
        audio = audio if audio.exists() else folder / "audio.flac"
        lyrics = folder / f"{name}_lyrics.txt"
        lyrics = lyrics if lyrics.exists() else folder / "lyrics.txt"
        if not audio.exists() or not lyrics.exists():
            print(f"⚠️ {name} 缺少音訊或歌詞檔，略過")
            continue
        print("\n🛠️", name)
        ok, log = run_lrc(audio, lyrics, folder / f"{name}.lrc", folder / f"{name}_segments.json")
        (folder / "lrc_log.txt").write_text(log, encoding="utf-8")
        print(log)
        _lrc = globals().get("LAST_LRC_INFO") or {}
        _hit = f"對上 {_lrc['matched']}/{_lrc['total']} 行" if _lrc.get("total") else ""
        print(f"✅ 完成 {_hit}" if ok else f"❌ 失敗 {_hit}，詳見 lrc_log.txt")


In [ ]:
# @title ⑥ 🎧 預覽
from IPython.display import Audio, display

PREVIEW_LIMIT = 5  # @param {type:"slider", min:1, max:20, step:1}

shown = 0
for folder in globals().get("generated_dirs", [])[:PREVIEW_LIMIT]:
    f = folder / f"{folder.name}.flac"
    if f.exists():
        print("🎵", folder.name)
        display(Audio(str(f)))
        shown += 1
if shown == 0:
    print("本批沒有可預覽的歌曲。")


In [ ]:
# @title ⑦ 🧹 釋放 GPU 記憶體（勾選才執行）
# @markdown 釋放後，下次生成前需重新執行 ④
RELEASE_GPU = False  # @param {type:"boolean"}

if not RELEASE_GPU:
    print("⏭️ 未勾選，保留已載入的 YuE2")
else:
    import gc
    pipe = None
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ 已釋放")


In [ ]:
# @title ⑧ 💾 備份到 Google Drive（自動，已備份會略過）
# 用官方的 save_pretrained() 匯出模型與 VAE，之後可完全離線載入。
# @markdown 重新備份（例如官方更新了模型）
FORCE_REBACKUP = False  # @param {type:"boolean"}

if not drive_ready():
    print("⏭️ 未啟用 Drive 備份（① 的 USE_DRIVE_BACKUP）")
else:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    # 程式碼
    if "yue2_code" in YUE2_RESTORED:
        print("✅ yue2_code：本次即由備份還原，略過")
    elif backup_done("yue2_code") and not FORCE_REBACKUP:
        print("✅ yue2_code：已備份")
    elif (YUE_REPO / "pyproject.toml").exists():
        print("💾 備份 yue2_code ……")
        backup_code_tar("yue2_code", YUE_REPO)
        print("✅ yue2_code：完成")

    # 模型 + VAE
    if globals().get("pipe") is None:
        print("⏭️ 模型尚未載入（請先執行 ④），略過模型備份")
    elif globals().get("YUE2_FROM_BACKUP"):
        print(f"✅ {YUE2_BACKUP_NAME}：本次即由備份載入，略過")
    elif backup_done(YUE2_BACKUP_NAME) and not FORCE_REBACKUP:
        print(f"✅ {YUE2_BACKUP_NAME}：已備份")
    else:
        print(f"💾 備份 {YUE2_BACKUP_NAME} ……")
        try:
            tmp = _begin(YUE2_BACKUP_NAME)
            pipe.save_pretrained(str(tmp / "pipeline"))
            _finish(tmp, YUE2_BACKUP_NAME)
            print(f"✅ 完成（{dir_size_gb(BACKUP_DIR / YUE2_BACKUP_NAME):.1f} GB）")
        except Exception as e:
            print(f"⚠️ 備份失敗：{e!r}（Drive 空間不足時常見）")

    print(f"\n備份位置：{BACKUP_DIR}")
    print("ℹ️ Drive 會在背景上傳，剛備份完時請等幾分鐘再中斷 Runtime。")
